# 🔐 AuthLab — Autenticação e Controle de Acesso
### OxeTech Lab · AccessLab Systems · laboratório prático para estudantes

**Uso exclusivamente educacional.** Todos os personagens, recursos, senhas e segredos deste laboratório são fictícios e descartáveis. Execute somente neste ambiente local em memória. Não insira credenciais pessoais nem conecte o laboratório a sistemas reais. **SecureAccess é didático, não é um sistema de produção.**

Texto, narrativa, exemplos e exercícios foram elaborados originalmente para este material. A identidade pedagógica acompanha o padrão informado do CriptoLab:

**📖 CONCEITO → 🧪 LABORATÓRIO → 🔎 ANÁLISE → 🎮 DESAFIO → 💡 DICA → 🛡️ BLUE TEAM**

Na **AccessLab Systems**, alunos registram relatórios de laboratório; analistas acompanham todos os relatórios; administradores também gerenciam usuários. Sua missão é separar identidade, segundo fator e permissão de acesso.

**Pré-requisitos:** funções, dicionários, condicionais e exceções em Python. Não é necessário usar celular, criar contas ou conhecer um framework web.

| Etapa | Tempo sugerido |
|---|---:|
| Preparação + Bloco 0 — AuthN x AuthZ | 25 min |
| Bloco 1 — Senhas | 35 min |
| Bloco 2 — MFA/TOTP | 40 min |
| Bloco 3 — Sessões/JWT | 35 min |
| Bloco 4 — Controle de acesso | 30 min |
| Bloco Final — SecureAccess + discussão | 55 min |
| **Total** | **220 min (3h40)** |

**Duas aulas:** divisão natural depois do Bloco 2 (100 min + 120 min). Na segunda aula, execute as células anteriores para reconstruir o estado; não é necessário repetir a discussão.

## Como usar

No **Google Colab**, envie este `.ipynb`, selecione um ambiente Python e use **Ambiente de execução → Executar tudo**. No **Jupyter**, use **Restart Kernel and Run All**; no **VS Code**, selecione um kernel Python com suporte Jupyter e use **Run All**. Recomenda-se Python 3.10 ou superior.

A instalação inicial baixa pacotes públicos do PyPI se faltarem. Depois disso, os exercícios não acessam a rede nem exigem contas de terceiros. A plataforma Colab pode exigir uma conta para executar; **o notebook também funciona localmente sem conta**. Em laboratório sem internet, o docente deve pré-instalar as versões indicadas abaixo.

Não há `input()` bloqueante, esperas reais, caminhos locais fixos nem chaves embutidas. O QR Code é opcional. Execute de cima para baixo: repetir células isoladas pode reutilizar estado de MFA. Para começar novamente, reinicie o kernel e execute tudo.

**Como aproveitar os desafios:** execute cada bloco, registre suas observações e implemente as extensões sugeridas em uma cópia do notebook. A versão principal demonstra um fluxo didático completo para servir de referência de estudo.


In [ ]:
# Central de adaptação: edite aqui para a próxima turma.
TURMA = "OxeTech Lab — Cibersegurança"
DOCENTE = "Fernando dos Santos Alves"
VERSAO_MATERIAL = "1.0.0"
GERAR_QR = False  # Segredo fictício; nunca use esse QR em contas pessoais.
ITERACOES_PBKDF2 = 600_000
INTERVALO_TOTP = 30
JANELA_TOTP = 1  # Passo anterior, atual e seguinte (não 1 segundo).
LIMITE_TENTATIVAS = 3
BLOQUEIO_SEGUNDOS = 60
JWT_TTL_SEGUNDOS = 300
ISSUER = "urn:accesslab:auth"
AUDIENCE = "urn:accesslab:relatorios"
SENHA_FICTICIA = "caju orbita farol laboratório 2026"
print(f"{TURMA} | Docente: {DOCENTE} | AuthLab {VERSAO_MATERIAL}")


In [ ]:
# Instala no Python do kernel atual, tanto no Colab quanto no Jupyter/VS Code.
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from importlib.util import find_spec

REQUISITOS = {"pyotp": "2.9.0", "PyJWT": "2.10.1"}
if GERAR_QR:
    REQUISITOS.update({"qrcode[pil]": "8.2"})

for pacote, versao_alvo in REQUISITOS.items():
    distribuicao = pacote.split("[")[0]
    try:
        presente = version(distribuicao) == versao_alvo
        if pacote == "qrcode[pil]" and find_spec("PIL") is None:
            presente = False
    except PackageNotFoundError:
        presente = False
    if not presente:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               f"{pacote}=={versao_alvo}"])
print("Dependências prontas:", {p: version(p.split("[")[0]) for p in REQUISITOS})
# Versões fixas tornam esta edição reproduzível. Docentes devem revisá-las periodicamente.


In [ ]:
import base64
import hashlib
import hmac
import json
import re
import secrets
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from collections import Counter
import pyotp
import jwt

TESTES = []

def verificar(nome, condicao):
    if not condicao:
        raise AssertionError(f"❌ Falha real: {nome}")
    TESTES.append(nome)
    print(f"✅ {nome}")

def esperar_erro(nome, tipo, acao):
    try:
        acao()
    except tipo as exc:
        TESTES.append(nome)
        print(f"✅ Recusa esperada: {nome} ({type(exc).__name__})")
        return exc
    raise AssertionError(f"❌ {nome}: a operação deveria ter sido recusada")

class Recusa(Exception):
    def __init__(self, status, mensagem):
        self.status = status
        super().__init__(mensagem)

def esperar_status(nome, status, acao):
    # Uma recusa com status diferente é uma falha real, não um teste aprovado.
    try:
        acao()
    except Recusa as exc:
        verificar(f"Recusa esperada: {nome} → {status}", exc.status == status)
        return
    raise AssertionError(f"❌ {nome}: acesso indevidamente permitido")

@dataclass
class RelogioSimulado:
    agora: float = 1_800_000_000.0
    def __call__(self):
        return self.agora
    def avancar(self, segundos):
        if segundos < 0:
            raise ValueError("O relógio de teste não pode voltar")
        self.agora += segundos

print("Auxiliares prontos. Só condições satisfeitas e recusas corretas recebem ✅.")


## Bloco 0 — AuthN x AuthZ
### 📖 CONCEITO

**Autenticação (AuthN)** verifica a identidade apresentada: “quem está solicitando?”. **Autorização (AuthZ)** decide se essa identidade pode executar determinada ação sobre determinado recurso: “o que pode fazer aqui?”. Entrar na plataforma não dá acesso a todos os relatórios.

| Situação na API fictícia | Resultado |
|---|---|
| Credencial ausente, inválida ou expirada | **401 Unauthorized**: falta autenticação válida |
| Identidade válida, mas permissão insuficiente | **403 Forbidden**: acesso recusado |
| Recurso inexistente, ou existência deliberadamente ocultada | **404 Not Found** |

Apesar do nome “Unauthorized”, 401 diz respeito à autenticação. Em HTTP real, respostas 401 devem incluir o desafio apropriado em `WWW-Authenticate` (por exemplo, Bearer). Aqui apenas simulamos os códigos; não há servidor HTTP. 404 pode evitar revelar a existência de um recurso, mas **não substitui** a checagem de autorização.

### 🧪 LABORATÓRIO
Antes de executar a célula seguinte, classifique mentalmente cada cenário como autenticação (AuthN) ou autorização (AuthZ). Ao final, explique por que uma identidade autenticada ainda pode receber uma recusa de acesso.


In [ ]:
cenarios = [
    "Conferir senha e segundo fator de Lia",
    "Decidir se Lia pode ler o relatório de Rui",
    "Validar assinatura, emissor e validade da credencial",
    "Exigir papel admin para gerenciar usuários",
]
print("Antes de continuar, classifique cada cenário como AuthN ou AuthZ:")
for indice, cenario in enumerate(cenarios, start=1):
    print(f"{indice}. {cenario}")


def porta_administrativa(identidade):
    if identidade is None:
        raise Recusa(401, "Autenticação necessária")
    if identidade["role"] != "admin":
        raise Recusa(403, "Permissão insuficiente")
    return "Painel administrativo fictício"

esperar_status("sem identidade", 401, lambda: porta_administrativa(None))
esperar_status("aluno autenticado", 403, lambda: porta_administrativa({"role": "aluno"}))
verificar("admin autorizado", "Painel" in porta_administrativa({"role": "admin"}))


### 🔎 ANÁLISE
O mesmo endpoint produz 401 ou 403 conforme a etapa que falhou. O dicionário da demonstração representa uma identidade **já autenticada pelo servidor**; receber `role` enviado livremente pelo cliente não autentica ninguém.

### 🎮 DESAFIO
Acrescente ao quiz: “token expirado”, “token válido sem permissão” e “recurso que não existe”. Classifique a etapa e o código de resposta. Escreva três testes com os auxiliares.

### 💡 DICA
Validade da credencial vem antes de permissão. Se uma resposta revelar algo sobre o recurso, pergunte se o solicitante deveria saber que ele existe.

### 🛡️ BLUE TEAM
- Um aumento de 401 sugere necessariamente um ataque? Que contexto falta?
- Muitos 403 para IDs diferentes após um login válido podem indicar o quê?
- Quais dados devem aparecer no log interno, mas não na resposta ao cliente?


## Bloco 1 — Senhas e autenticação segura
### 📖 CONCEITO

Não armazenamos a senha nem um SHA-256 puro dela. Uma função de derivação de chave para senhas aumenta o custo de cada tentativa offline. Usaremos **PBKDF2-HMAC-SHA256 com 600.000 iterações**, salt aleatório individual de 16 bytes e resultado de 32 bytes. O custo deve ser medido e revisto no hardware do serviço. Argon2id é uma alternativa moderna com custo de memória; PBKDF2 permite uma demonstração com a biblioteca padrão.

O **salt não é secreto**: ele impede que senhas iguais tenham registros iguais e reduz o reaproveitamento de tabelas pré-computadas. Guardamos salt, algoritmo, custo e hash juntos. A comparação usa `hmac.compare_digest`; isso não torna todo o fluxo imune a análise de tempo.

A política didática permite frases de 15 a 128 caracteres e bloqueia uma pequena lista fictícia. Não exigimos combinações arbitrárias de símbolos: comprimento, unicidade, gerenciador de senhas, bloqueio de senhas comprometidas e MFA são mais úteis que complexidade isolada. A lista local **não é** uma base completa de senhas vazadas.

### 🧪 LABORATÓRIO
Cadastre duas pessoas com a mesma senha fictícia e compare os registros, sem imprimir a senha ou o hash.


In [ ]:
SENHAS_PROIBIDAS_DEMO = {"123456789012345", "senha senha senha"}

def validar_politica(senha):
    if not isinstance(senha, str) or not 15 <= len(senha) <= 128:
        raise ValueError("Use uma frase fictícia de 15 a 128 caracteres")
    if senha.casefold() in SENHAS_PROIBIDAS_DEMO:
        raise ValueError("Senha presente na lista de bloqueio didática")

def derivar_senha(senha):
    validar_politica(senha)
    salt = secrets.token_bytes(16)
    resumo = hashlib.pbkdf2_hmac("sha256", senha.encode("utf-8"), salt,
                                 ITERACOES_PBKDF2, dklen=32)
    return {"algoritmo": "pbkdf2-sha256", "iteracoes": ITERACOES_PBKDF2,
            "salt": salt, "hash": resumo}

def conferir_senha(senha, registro):
    if not isinstance(senha, str) or len(senha) > 128:
        return False
    if registro["algoritmo"] != "pbkdf2-sha256":
        raise ValueError("Formato de armazenamento não suportado")
    calculado = hashlib.pbkdf2_hmac("sha256", senha.encode("utf-8"),
                                  registro["salt"], registro["iteracoes"], dklen=32)
    return hmac.compare_digest(calculado, registro["hash"])

contas_demo = {nome: derivar_senha(SENHA_FICTICIA) for nome in ("lia", "rui")}
verificar("salt individual", contas_demo["lia"]["salt"] != contas_demo["rui"]["salt"])
verificar("mesma senha, hashes diferentes", contas_demo["lia"]["hash"] != contas_demo["rui"]["hash"])
verificar("senha correta", conferir_senha(SENHA_FICTICIA, contas_demo["lia"]))
verificar("senha errada recusada", not conferir_senha("errada", contas_demo["lia"]))
verificar("senha em texto não armazenada", set(contas_demo["lia"]) == {"algoritmo", "iteracoes", "salt", "hash"})
esperar_erro("frase curta", ValueError, lambda: derivar_senha("curta"))
esperar_erro("senha bloqueada", ValueError, lambda: derivar_senha("senha senha senha"))


**Limite de tentativas em tempo simulado.** O contador é indexado por conta e não é zerado por cada tentativa bloqueada. Três falhas impõem uma pausa; após a pausa, uma nova tentativa é possível. O relógio é uma dependência do **servidor**, nunca um parâmetro fornecido pelo usuário.

Esta simulação é sequencial e em memória: não implemente um serviço copiando-a. Em produção seriam necessários estado compartilhado e operações atômicas, limites por conta/origem/dispositivo, expiração de entradas e monitoramento. Bloqueios por conta também podem ser abusados para negação de serviço. Mensagens genéricas e um hash substituto ajudam a reduzir enumeração, mas não garantem tempo idêntico.


In [ ]:
class Limitador:
    def __init__(self, relogio, limite=LIMITE_TENTATIVAS, pausa=BLOQUEIO_SEGUNDOS):
        self.relogio, self.limite, self.pausa = relogio, limite, pausa
        self.estado = {}

    def permitir(self, conta):
        falhas, ate = self.estado.get(conta, (0, 0))
        if self.relogio() < ate:
            return False
        if ate:
            self.estado.pop(conta, None)
        return True

    def falhou(self, conta):
        falhas, _ = self.estado.get(conta, (0, 0))
        falhas += 1
        ate = self.relogio() + self.pausa if falhas >= self.limite else 0
        self.estado[conta] = (falhas, ate)

    def sucesso(self, conta):
        self.estado.pop(conta, None)

relogio_senhas = RelogioSimulado()
lim_senhas = Limitador(relogio_senhas)
for _ in range(LIMITE_TENTATIVAS):
    verificar("tentativa dentro do limite", lim_senhas.permitir("lia"))
    lim_senhas.falhou("lia")
verificar("limite atingido: recusa", not lim_senhas.permitir("lia"))
verificar("outra conta tem contador próprio", lim_senhas.permitir("rui"))
relogio_senhas.avancar(BLOQUEIO_SEGUNDOS)
verificar("nova tentativa após pausa simulada", lim_senhas.permitir("lia"))


### 🔎 ANÁLISE
O salt resolve repetição de registros; o custo da KDF reduz tentativas offline; o limitador reduz tentativas online. São controles diferentes. Alterar `ITERACOES_PBKDF2` não muda registros antigos: uma aplicação real precisaria de migração ou rehash após login.

### 🎮 DESAFIO
Crie um `Limitador` separado com limite 2 e pausa 90. Teste falha, bloqueio, tentativa aos 89 segundos e liberação aos 90. Acrescente ao menos uma frase à lista fictícia de bloqueio e teste a recusa. Não reduza o custo da KDF para “acelerar” uma aplicação real.

### 💡 DICA
Use outro `RelogioSimulado` e instâncias novas, evitando que o desafio altere os testes seguintes. O acesso é recusado enquanto `agora < ate`.

### 🛡️ BLUE TEAM
- Como distinguir erros comuns de senha de password spraying distribuído?
- Por que o bloqueio só por IP falha em redes compartilhadas e ataques distribuídos?
- Que informações um alerta precisa conter sem registrar senhas ou hashes?


## Bloco 2 — MFA com TOTP
### 📖 CONCEITO

**Conhecimento:** algo que você sabe (senha). **Posse:** algo que você possui (autenticador com segredo). **Inerência:** algo que você é (característica biométrica). Duas senhas continuam sendo um só tipo de fator.

TOTP deriva um código de um segredo compartilhado e de um passo de tempo, aqui de 30 segundos. O servidor também precisa do segredo para validar; não basta guardar um hash dele como fazemos com senhas. Em produção, o segredo precisa de proteção em repouso, acesso restrito, provisionamento autenticado e recuperação segura.

`valid_window=1` aceita três passos: anterior, atual e seguinte. Isso tolera desvio de relógio, mas amplia a oportunidade de uso. **PyOTP verificar um código não o consome**: impedir replay exige estado no servidor. TOTP não resiste a phishing em tempo real; passkeys/WebAuthn são uma extensão pertinente.

Neste notebook, “telefone” e servidor são objetos no mesmo processo para dispensar dispositivos. Isso **simula** fatores separados; não constitui MFA real no computador do aluno.

### 🧪 LABORATÓRIO
Usaremos tempo controlado para demonstrar expiração sem esperar 30 segundos. O valor de tempo não é aceito de um cliente.


In [ ]:
def novo_segredo_demo(instante):
    # Evita colisões raras entre códigos dos passos usados nos testes didáticos.
    # Continua aleatório; em produção basta pyotp.random_base32().
    while True:
        segredo = pyotp.random_base32()
        gerador = pyotp.TOTP(segredo, interval=INTERVALO_TOTP)
        amostra = [gerador.at(instante + i * INTERVALO_TOTP) for i in range(-2, 6)]
        if len(set(amostra)) == len(amostra):
            return segredo

t_mfa = RelogioSimulado()
segredo_demo = novo_segredo_demo(t_mfa())
totp_demo = pyotp.TOTP(segredo_demo, interval=INTERVALO_TOTP)
codigo_demo = totp_demo.at(t_mfa())
verificar("TOTP atual válido", totp_demo.verify(codigo_demo, for_time=t_mfa()))
verificar("sem controle de replay, biblioteca aceita novamente",
          totp_demo.verify(codigo_demo, for_time=t_mfa()))
verificar("passo seguinte rejeita com janela zero",
          not totp_demo.verify(codigo_demo, for_time=t_mfa() + INTERVALO_TOTP))
verificar("janela 1 aceita passo anterior",
          totp_demo.verify(codigo_demo, for_time=t_mfa() + INTERVALO_TOTP, valid_window=1))
verificar("dois passos depois, código fora da janela 1",
          not totp_demo.verify(codigo_demo, for_time=t_mfa() + 2 * INTERVALO_TOTP, valid_window=1))

if GERAR_QR:
    import qrcode
    from IPython.display import display
    uri = totp_demo.provisioning_uri(name="lia@accesslab.example", issuer_name="AccessLab Systems — DEMO")
    display(qrcode.make(uri).get_image())
    print("QR fictício: quem o lê obtém o segredo. Não compartilhe QRs de contas reais.")
else:
    print("QR opcional desativado. Todos os testes funcionam sem celular.")


In [ ]:
class VerificadorTOTP:
    def __init__(self, segredo, relogio, janela=JANELA_TOTP):
        self.totp = pyotp.TOTP(segredo, interval=INTERVALO_TOTP)
        self.relogio, self.janela = relogio, janela
        self.ultimo_passo = -1

    def consumir(self, codigo):
        if not isinstance(codigo, str) or not re.fullmatch(r"[0-9]{6}", codigo):
            return False
        atual = int(self.relogio() // INTERVALO_TOTP)
        correspondencias = [p for p in range(atual - self.janela, atual + self.janela + 1)
                            if p >= 0 and hmac.compare_digest(self.totp.at(p * INTERVALO_TOTP), codigo)]
        if not correspondencias or min(correspondencias) <= self.ultimo_passo:
            return False
        self.ultimo_passo = max(correspondencias)
        return True

def codigo_invalido(gerador, instante, janela=JANELA_TOTP):
    # Um número aleatório fixo como 000000 poderia, raramente, ser válido.
    validos = {gerador.at(instante + i * INTERVALO_TOTP) for i in range(-janela, janela + 1)}
    return next(f"{i:06d}" for i in range(100) if f"{i:06d}" not in validos)

verificador_mfa = VerificadorTOTP(segredo_demo, t_mfa)
verificar("primeiro uso consumido", verificador_mfa.consumir(codigo_demo))
verificar("replay recusado", not verificador_mfa.consumir(codigo_demo))
t_mfa.avancar(INTERVALO_TOTP)
verificar("passo novo aceito", verificador_mfa.consumir(totp_demo.at(t_mfa())))
verificar("formato inválido recusado", not verificador_mfa.consumir("abc"))


### 🔎 ANÁLISE
O código do passo anterior pode estar “velho” e ainda ser aceito por tolerância. Aceitar novamente um código consumido é outra questão: **replay**. Guardar o último passo aceito também rejeita passos anteriores, mesmo dentro da janela. Aceitar um passo adiantado exige esperar um passo posterior para o próximo login.

O consumo precisa ser atômico em produção; duas requisições simultâneas não podem aceitar o mesmo passo. Reiniciar o estado em memória apaga a proteção. Além disso, MFA precisa de limite de tentativas por conta que sobreviva à criação de novos desafios de login.

### 🎮 DESAFIO
Execute o ponto de partida abaixo. Altere o limite para 2 e teste uma tentativa bloqueada mesmo com o código correto. Depois da pausa, valide um código novo e conte eventos por resultado. Os logs não podem conter código, segredo, senha ou token.

### 💡 DICA
Primeiro consulte o limitador; depois valide/consuma o código. Registre apenas resultado, conta fictícia, etapa e horário. O SecureAccess integrará esse fluxo com um desafio temporário vinculado à senha já verificada.


In [ ]:
relogio_desafio = RelogioSimulado()
segredo_desafio = novo_segredo_demo(relogio_desafio())
v_desafio = VerificadorTOTP(segredo_desafio, relogio_desafio)
l_desafio = Limitador(relogio_desafio)
eventos_desafio = []

def tentar_mfa_demo(codigo):
    if not l_desafio.permitir("lia"):
        resultado = "bloqueado"
    elif v_desafio.consumir(codigo):
        l_desafio.sucesso("lia")
        resultado = "permitido"
    else:
        l_desafio.falhou("lia")
        resultado = "recusado"
    eventos_desafio.append({"conta": "lia", "etapa": "mfa", "resultado": resultado,
                            "instante": relogio_desafio()})
    return resultado

errado_mfa = codigo_invalido(v_desafio.totp, relogio_desafio())
for _ in range(LIMITE_TENTATIVAS):
    verificar("TOTP errado recusado", tentar_mfa_demo(errado_mfa) == "recusado")
verificar("TOTP correto bloqueado após limite",
          tentar_mfa_demo(v_desafio.totp.at(relogio_desafio())) == "bloqueado")
relogio_desafio.avancar(BLOQUEIO_SEGUNDOS)
verificar("TOTP novo após pausa", tentar_mfa_demo(v_desafio.totp.at(relogio_desafio())) == "permitido")
print("Eventos por resultado:", dict(Counter(e["resultado"] for e in eventos_desafio)))


### 🛡️ BLUE TEAM
- Muitos desafios iniciados e poucos MFA concluídos sugerem quais hipóteses?
- Como detectar tentativas de replay sem guardar os códigos nos logs?
- Como recuperar uma conta sem transformar o suporte técnico em um caminho para contornar MFA?


## Bloco 3 — Sessões e Tokens
### 📖 CONCEITO

| Modelo | Como reconhece a sessão | Revogação e riscos |
|---|---|---|
| **Stateful** | Cliente apresenta ID aleatório; servidor consulta estado | Remover a sessão pode encerrar o acesso imediatamente; proteger armazenamento e cookie |
| **Token stateless** | Servidor valida assinatura e claims sem consultar uma sessão | Token pode valer até expirar; revogação antecipada exige estado ou outra estratégia |

JWT é um formato, não uma solução completa de sessão. O JWT assinado usado aqui (**JWS**) tem cabeçalho, payload e assinatura. Base64url é codificação, **não criptografia**. Não coloque segredos no payload. Quem rouba um bearer token pode usá-lo durante sua validade; MFA no login não impede esse uso.

Usaremos **HS256** com chave aleatória de 32 bytes, criada só para esta execução. Em HMAC, quem valida com a chave também consegue assinar; arquiteturas com verificadores independentes podem preferir assinatura assimétrica. A allowlist vem da configuração do servidor, nunca do cabeçalho recebido.

Claims: `sub` identifica a conta, `iat` indica emissão, `exp` expiração, `iss` emissor, `aud` destinatário. `role` e `scope` são convenções da aplicação; `jti` identifica o token. Validar presença não basta: é preciso validar assinatura, valores, tipos e regras de autorização.

### 🧪 LABORATÓRIO
O relógio de JWT é o UTC real usado pela biblioteca; tokens expirados de teste são assinados com datas no passado. Não desabilitamos a validação de expiração. Os relógios simulados dos outros blocos controlam apenas TOTP, desafios e limitadores.


In [ ]:
PAPEIS = {"aluno", "analista", "admin"}
SCOPES_POR_ROLE = {"aluno": "relatorio:ler", "analista": "relatorio:ler",
                   "admin": "relatorio:ler usuarios:gerenciar"}
CLAIMS_OBRIGATORIAS = ["sub", "iat", "exp", "iss", "aud", "role", "scope", "jti"]

def emitir_jwt(chave, sub, role):
    agora = int(time.time())
    claims = {"sub": sub, "iat": agora, "exp": agora + JWT_TTL_SEGUNDOS,
              "iss": ISSUER, "aud": AUDIENCE, "role": role,
              "scope": SCOPES_POR_ROLE[role], "jti": secrets.token_urlsafe(16)}
    return jwt.encode(claims, chave, algorithm="HS256")

def validar_jwt(token, chave):
    claims = jwt.decode(token, chave, algorithms=["HS256"], issuer=ISSUER,
                        audience=AUDIENCE, leeway=0,
                        options={"require": CLAIMS_OBRIGATORIAS, "strict_aud": True})
    if (not isinstance(claims["sub"], str) or not claims["sub"]
            or type(claims["iat"]) is not int or type(claims["exp"]) is not int
            or claims["exp"] <= claims["iat"]
            or not isinstance(claims["role"], str) or claims["role"] not in PAPEIS
            or not isinstance(claims["scope"], str)
            or claims["scope"] != SCOPES_POR_ROLE[claims["role"]]
            or not isinstance(claims["jti"], str) or not claims["jti"]):
        raise jwt.InvalidTokenError("Claims da aplicação inválidas")
    return claims

def ler_payload_para_inspecao(token):
    # Apenas visualização. NUNCA usar este resultado para autenticar/autorizar.
    segmento = token.split(".")[1]
    return json.loads(base64.urlsafe_b64decode(segmento + "=" * (-len(segmento) % 4)))

def adulterar_payload_demo(token):
    cabecalho, _, assinatura = token.split(".")
    claims = ler_payload_para_inspecao(token)
    claims["role"] = "admin"
    segmento = base64.urlsafe_b64encode(json.dumps(claims).encode()).rstrip(b"=").decode()
    return f"{cabecalho}.{segmento}.{assinatura}"

chave_demo = secrets.token_bytes(32)
token_demo = emitir_jwt(chave_demo, "lia", "aluno")
verificar("token válido", validar_jwt(token_demo, chave_demo)["sub"] == "lia")
print("Payload legível, sem validar (somente inspeção):",
      {k: v for k, v in ler_payload_para_inspecao(token_demo).items() if k != "jti"})
esperar_erro("payload alterado invalida assinatura", jwt.InvalidSignatureError,
             lambda: validar_jwt(adulterar_payload_demo(token_demo), chave_demo))


In [ ]:
# Fixtures locais assinadas pelo próprio laboratório para isolar cada validação.
# Não são uma interface de emissão exposta a usuários.
def fixture_token(chave, claims_base, alteracoes=None, remover=(), algoritmo="HS256"):
    claims = dict(claims_base)
    claims.update(alteracoes or {})
    for campo in remover:
        claims.pop(campo, None)
    return jwt.encode(claims, chave, algorithm=algoritmo)

base_claims = validar_jwt(token_demo, chave_demo)
passado = int(time.time()) - 600
casos_jwt = [
    ("token expirado", {"iat": passado, "exp": passado + 30}, (), jwt.ExpiredSignatureError),
    ("emissor incorreto", {"iss": "urn:outro"}, (), jwt.InvalidIssuerError),
    ("audience incorreta", {"aud": "urn:outro"}, (), jwt.InvalidAudienceError),
    ("exp ausente", {}, ("exp",), jwt.MissingRequiredClaimError),
    ("role ausente", {}, ("role",), jwt.MissingRequiredClaimError),
    ("role desconhecida", {"role": "superuser"}, (), jwt.InvalidTokenError),
    ("scope incompatível", {"scope": "usuarios:gerenciar"}, (), jwt.InvalidTokenError),
    ("iat no futuro", {"iat": int(time.time()) + 3600}, (), jwt.ImmatureSignatureError),
]
for nome, alteracoes, remover, erro in casos_jwt:
    fixture = fixture_token(chave_demo, base_claims, alteracoes, remover)
    esperar_erro(nome, erro, lambda fixture=fixture: validar_jwt(fixture, chave_demo))
token_outro_alg = fixture_token(chave_demo, base_claims, algoritmo="HS384")
esperar_erro("algoritmo fora da allowlist", jwt.InvalidAlgorithmError,
             lambda: validar_jwt(token_outro_alg, chave_demo))
esperar_erro("outra chave não valida assinatura", jwt.InvalidSignatureError,
             lambda: validar_jwt(token_demo, secrets.token_bytes(32)))


In [ ]:
# Contraponto stateful: estado de sessão mantido no servidor.
sessoes_demo = {}
id_sessao = secrets.token_urlsafe(32)
sessoes_demo[id_sessao] = {"sub": "lia", "exp": time.time() + 300}

def validar_sessao(id_recebido):
    sessao = sessoes_demo.get(id_recebido)
    if not sessao or sessao["exp"] <= time.time():
        raise Recusa(401, "Sessão inválida")
    return sessao["sub"]

verificar("sessão reconhecida pelo estado", validar_sessao(id_sessao) == "lia")
del sessoes_demo[id_sessao]
esperar_status("sessão removida", 401, lambda: validar_sessao(id_sessao))
verificar("remover sessão não revoga JWT independente", validar_jwt(token_demo, chave_demo)["sub"] == "lia")


### 🔎 ANÁLISE
Ler o payload sem chave demonstra ausência de confidencialidade. Alterá-lo mantém a aparência de um JWT, mas rompe sua assinatura. Tokens assinados com issuer ou audience errados também são recusados. Decodificar não é validar.

Uma role assinada pode ficar desatualizada depois de uma mudança de permissão. Expiração curta reduz a janela; consulta a estado, versionamento ou revogação podem ser necessários. Tokens não são automaticamente mais seguros que sessões. Em aplicações web, armazenamento e transporte exigem TLS, cookies adequados quando usados (`Secure`, `HttpOnly`, `SameSite`) e tratamento de CSRF/XSS conforme a arquitetura.

### 🎮 DESAFIO
Adicione um teste de `aud` ausente e outro de sessão expirada. Explique o que precisa mudar para encerrar acesso imediatamente após remover uma role. Mantenha a validação de assinatura e expiração ativa.

### 💡 DICA
Reutilize `fixture_token(..., remover=("aud",))` para gerar uma credencial local de teste. Para sessão, ajuste somente a data no dicionário fictício.

### 🛡️ BLUE TEAM
- Como investigar uso de bearer token roubado sem registrar o token completo?
- Qual o impacto de uma chave HS256 comprometida? Como rotacioná-la?
- Que diferença há entre falha de assinatura e expiração normal nos indicadores internos?


## Bloco 4 — Controle de acesso
### 📖 CONCEITO

**RBAC** atribui permissões a papéis. Aqui: `aluno` lê os próprios relatórios; `analista` lê todos; `admin` lê todos e gerencia usuários. Permitir analistas em todos os relatórios é uma **decisão explícita deste cenário**, não uma recomendação universal.

**ABAC** avalia atributos da identidade, do recurso e do contexto (por exemplo, departamento, classificação e turno). **Propriedade do recurso** relaciona a identidade autenticada ao dono do objeto. RBAC e propriedade podem atuar juntos: ter a permissão “ler relatório” não basta para ler qualquer relatório.

Broken Access Control ocorre quando controles de acesso falham. IDOR é um caso em que a referência a um objeto é usada sem a autorização adequada. IDs difíceis de adivinhar não substituem checagens por objeto. Esta simulação vulnerável existe apenas em um dicionário local; não há alvos externos.

### 🧪 LABORATÓRIO
Primeiro observe o defeito controlado; depois use exclusivamente a função corrigida.


In [ ]:
RECURSOS_DEMO = {
    "rel-101": {"id": "rel-101", "dono": "lia", "conteudo": "Relatório fictício de Lia"},
    "rel-202": {"id": "rel-202", "dono": "rui", "conteudo": "Relatório fictício de Rui"},
}

def ler_inseguro_apenas_demonstracao(identidade, recurso_id):
    if identidade is None:
        raise Recusa(401, "Autenticação necessária")
    return RECURSOS_DEMO[recurso_id]  # DEFEITO INTENCIONAL: nenhuma autorização por objeto.

lia_autenticada = {"sub": "lia", "role": "aluno", "scope": "relatorio:ler"}
vazamento = ler_inseguro_apenas_demonstracao(lia_autenticada, "rel-202")
verificar("defeito reproduzido em memória: relatório de outro dono", vazamento["dono"] != "lia")
print("Acesso indevido observado. A partir daqui, somente a versão corrigida.")

def autorizar_leitura(identidade, recurso_id, recursos):
    # Precondição: identidade veio de validação de sessão/token no servidor.
    if identidade is None:
        raise Recusa(401, "Autenticação necessária")
    if identidade["role"] not in PAPEIS or "relatorio:ler" not in identidade["scope"].split():
        raise Recusa(403, "Permissão insuficiente")
    recurso = recursos.get(recurso_id)
    if recurso is None:
        raise Recusa(404, "Recurso não encontrado")
    if identidade["role"] == "aluno" and recurso["dono"] != identidade["sub"]:
        raise Recusa(404, "Recurso não encontrado")  # Oculta existência para não proprietários.
    return dict(recurso)

verificar("aluno lê o próprio recurso", autorizar_leitura(lia_autenticada, "rel-101", RECURSOS_DEMO)["dono"] == "lia")
esperar_status("outro proprietário (existência ocultada)", 404,
               lambda: autorizar_leitura(lia_autenticada, "rel-202", RECURSOS_DEMO))
esperar_status("ID inexistente", 404, lambda: autorizar_leitura(lia_autenticada, "ausente", RECURSOS_DEMO))
esperar_status("sem autenticação", 401, lambda: autorizar_leitura(None, "rel-101", RECURSOS_DEMO))
esperar_status("sem scope", 403,
               lambda: autorizar_leitura(dict(lia_autenticada, scope=""), "rel-101", RECURSOS_DEMO))
for papel in ("analista", "admin"):
    identidade = {"sub": "operador", "role": papel, "scope": SCOPES_POR_ROLE[papel]}
    verificar(f"{papel} lê recurso conforme política",
              autorizar_leitura(identidade, "rel-202", RECURSOS_DEMO)["dono"] == "rui")


### 🔎 ANÁLISE
A correção usa o `sub` da identidade validada, não um campo `dono` enviado pelo cliente. 403 significa que a ação é proibida pelo papel/scope. Para aluno sem propriedade e para objeto inexistente, a resposta pública é o mesmo 404. Isso não garante que todo canal lateral desapareça.

### 🎮 DESAFIO
Crie três relatórios adicionais e teste cada papel. Acrescente uma regra ABAC fictícia: um analista só pode ler relatórios do mesmo departamento. Teste tanto a permissão quanto a recusa, preservando o comportamento do aluno.

### 💡 DICA
Antes de alterar código, escreva uma matriz “papel × recurso × resultado”. Adote negação por padrão para papéis ou ações desconhecidas. A implementação de ABAC fica como extensão; a base usa RBAC + propriedade.

### 🛡️ BLUE TEAM
- Quais sequências de 404 podem sugerir tentativa de enumeração de objetos?
- Como auditar a concessão de papéis privilegiados?
- Um UUID resolveria o defeito da função vulnerável? Por quê?


## Bloco Final — SecureAccess
### 📖 CONCEITO

Vamos integrar os controles em um fluxo de duas etapas:

**Cadastro → senha válida → desafio temporário de MFA → TOTP válido e não reutilizado → JWT → validação → role/scope → propriedade → recurso.**

O cadastro público cria sempre `aluno`. Papéis privilegiados da demonstração são provisionados por um administrador confiável do ambiente, fora do cadastro. O cadastro entrega um segredo fictício para simular o provisionamento do autenticador; a ativação/recuperação de MFA completa fica fora deste mini sistema.

Um desafio aleatório dura 120 segundos, fica vinculado à conta e é consumido no sucesso. Novo login substitui o desafio anterior da conta, mas **não zera falhas de MFA**. Há limite por conta na senha e no segundo fator. O mesmo passo TOTP não pode concluir dois logins.

Logs usam campos definidos e não recebem senhas, hashes, segredos TOTP, códigos, desafios ou tokens. Respostas de senha errada e conta inexistente são genéricas. Neste ambiente didático, o processo inteiro está sob controle do aluno; atributos Python não são uma barreira de segurança contra quem executa o notebook.

### 🧪 LABORATÓRIO
A classe usa as funções revisadas nos blocos anteriores. Trata-se de um modelo **híbrido**: JWT assinado, mas conta e papel atual são consultados em memória a cada acesso. Mudanças de papel invalidam a credencial antiga. Não há revogação individual por `jti` nesta versão.


In [ ]:
class SecureAccess:
    def __init__(self, relogio):
        self.relogio = relogio
        self._chave = secrets.token_bytes(32)
        self._usuarios, self._pendentes, self.eventos = {}, {}, []
        self._recursos = {k: dict(v) for k, v in RECURSOS_DEMO.items()}
        self._lim_senha = Limitador(relogio)
        self._lim_mfa = Limitador(relogio)
        self._registro_dummy = derivar_senha("conta inexistente frase ficticia")

    def _log(self, etapa, resultado, usuario="desconhecido", recurso=None):
        self.eventos.append({"instante": self.relogio(), "etapa": etapa,
                             "resultado": resultado, "usuario": usuario, "recurso": recurso})

    def cadastrar(self, usuario, senha):
        if not isinstance(usuario, str) or not re.fullmatch(r"[a-z][a-z0-9_-]{2,31}", usuario):
            raise ValueError("Identificador fictício: 3 a 32 caracteres minúsculos")
        if usuario in self._usuarios:
            raise ValueError("Cadastro indisponível")
        registro = derivar_senha(senha)
        segredo = novo_segredo_demo(self.relogio())
        self._usuarios[usuario] = {"senha": registro, "role": "aluno",
                                   "mfa": VerificadorTOTP(segredo, self.relogio)}
        self._log("cadastro", "permitido", usuario)
        return segredo  # Canal local fictício de provisionamento, nunca um log.

    def iniciar_login(self, usuario, senha):
        if not isinstance(usuario, str) or not re.fullmatch(r"[a-z][a-z0-9_-]{2,31}", usuario):
            self._log("senha", "recusado")
            raise Recusa(401, "Credenciais inválidas")
        if not self._lim_senha.permitir(usuario):
            self._log("senha", "bloqueado", usuario)
            raise Recusa(429, "Aguarde antes de tentar novamente")
        conta = self._usuarios.get(usuario)
        registro = conta["senha"] if conta else self._registro_dummy
        senha_ok = conferir_senha(senha, registro)
        if not senha_ok or conta is None:
            self._lim_senha.falhou(usuario)
            self._log("senha", "recusado", usuario)
            raise Recusa(401, "Credenciais inválidas")
        self._lim_senha.sucesso(usuario)
        # Remove expirados e substitui qualquer desafio anterior da conta.
        self._pendentes = {k: v for k, v in self._pendentes.items()
                           if v["exp"] > self.relogio() and v["usuario"] != usuario}
        desafio = secrets.token_urlsafe(32)
        self._pendentes[desafio] = {"usuario": usuario, "exp": self.relogio() + 120}
        self._log("senha", "permitido", usuario)
        return desafio  # Não é JWT e não permite ler recursos.

    def concluir_mfa(self, desafio, codigo):
        pendente = self._pendentes.get(desafio) if isinstance(desafio, str) else None
        if not pendente or pendente["exp"] <= self.relogio():
            if isinstance(desafio, str):
                self._pendentes.pop(desafio, None)
            self._log("mfa", "desafio_invalido")
            raise Recusa(401, "Autenticação incompleta ou expirada")
        usuario = pendente["usuario"]
        if not self._lim_mfa.permitir(usuario):
            self._log("mfa", "bloqueado", usuario)
            raise Recusa(429, "Aguarde antes de tentar novamente")
        conta = self._usuarios[usuario]
        if not conta["mfa"].consumir(codigo):
            self._lim_mfa.falhou(usuario)
            self._log("mfa", "recusado", usuario)
            raise Recusa(401, "Autenticação incompleta ou expirada")
        self._lim_mfa.sucesso(usuario)
        del self._pendentes[desafio]
        token = emitir_jwt(self._chave, usuario, conta["role"])
        self._log("mfa", "permitido", usuario)
        self._log("token", "emitido", usuario)
        return token

    def autenticar(self, token):
        try:
            claims = validar_jwt(token, self._chave)
        except jwt.InvalidTokenError:
            self._log("token", "recusado")
            raise Recusa(401, "Credencial inválida") from None
        conta = self._usuarios.get(claims["sub"])
        if conta is None or conta["role"] != claims["role"]:
            self._log("token", "conta_ou_papel_alterado")
            raise Recusa(401, "Credencial inválida")
        self._log("token", "permitido", claims["sub"])
        return claims

    def ler_relatorio(self, token, recurso_id):
        identidade = self.autenticar(token)
        try:
            recurso = autorizar_leitura(identidade, recurso_id, self._recursos)
        except Recusa as exc:
            self._log("autorizacao", f"recusado_{exc.status}", identidade["sub"], recurso_id)
            raise
        self._log("autorizacao", "permitido", identidade["sub"], recurso_id)
        return recurso

    def listar_usuarios(self, token):
        identidade = self.autenticar(token)
        if identidade["role"] != "admin" or "usuarios:gerenciar" not in identidade["scope"].split():
            self._log("autorizacao", "recusado_403", identidade["sub"])
            raise Recusa(403, "Permissão insuficiente")
        self._log("autorizacao", "permitido", identidade["sub"])
        return sorted(self._usuarios)  # Nunca retorna registros de senha/MFA.

print("SecureAccess pronto: senha → desafio → MFA → JWT → autorização.")


In [ ]:
relogio_app = RelogioSimulado()
app = SecureAccess(relogio_app)
autenticadores_ficticios = {}
for usuario in ("lia", "rui", "bia", "eva"):
    segredo = app.cadastrar(usuario, SENHA_FICTICIA)
    autenticadores_ficticios[usuario] = pyotp.TOTP(segredo, interval=INTERVALO_TOTP)

# Configuração administrativa confiável do cenário. Não é entrada do cadastro.
app._usuarios["bia"]["role"] = "analista"
app._usuarios["eva"]["role"] = "admin"
verificar("cadastro público atribui aluno", app._usuarios["lia"]["role"] == "aluno")
esperar_erro("cadastro não aceita role fornecida pelo cliente", TypeError,
             lambda: app.cadastrar("intruso", SENHA_FICTICIA, role="admin"))
esperar_erro("cadastro duplicado recusado", ValueError, lambda: app.cadastrar("lia", SENHA_FICTICIA))
verificar("salt individual no sistema", app._usuarios["lia"]["senha"]["salt"] != app._usuarios["rui"]["senha"]["salt"])

esperar_status("senha errada", 401, lambda: app.iniciar_login("lia", "senha errada"))
esperar_status("conta inexistente", 401, lambda: app.iniciar_login("fantasma", SENHA_FICTICIA))
desafio_lia = app.iniciar_login("lia", SENHA_FICTICIA)
verificar("senha sozinha não emite JWT", not any(e["etapa"] == "token" and e["resultado"] == "emitido" for e in app.eventos))
esperar_status("desafio não serve como token", 401, lambda: app.ler_relatorio(desafio_lia, "rel-101"))
esperar_status("sem etapa de senha", 401, lambda: app.concluir_mfa("desafio-inexistente", "000000"))
otp_errado = codigo_invalido(autenticadores_ficticios["lia"], relogio_app())
esperar_status("TOTP inválido", 401, lambda: app.concluir_mfa(desafio_lia, otp_errado))
codigo_lia = autenticadores_ficticios["lia"].at(relogio_app())
token_lia = app.concluir_mfa(desafio_lia, codigo_lia)
verificar("JWT após MFA pertence a Lia", app.autenticar(token_lia)["sub"] == "lia")
esperar_status("desafio consumido não pode ser repetido", 401, lambda: app.concluir_mfa(desafio_lia, codigo_lia))
outro_desafio = app.iniciar_login("lia", SENHA_FICTICIA)
esperar_status("replay em novo login", 401, lambda: app.concluir_mfa(outro_desafio, codigo_lia))
verificar("próprio recurso permitido", app.ler_relatorio(token_lia, "rel-101")["dono"] == "lia")
esperar_status("recurso de outro usuário", 404, lambda: app.ler_relatorio(token_lia, "rel-202"))
esperar_status("role inadequada para administrar", 403, lambda: app.listar_usuarios(token_lia))
esperar_status("recurso ausente", 404, lambda: app.ler_relatorio(token_lia, "rel-999"))
esperar_status("token ausente", 401, lambda: app.ler_relatorio(None, "rel-101"))
esperar_status("token adulterado", 401, lambda: app.ler_relatorio(adulterar_payload_demo(token_lia), "rel-101"))

claims_lia = validar_jwt(token_lia, app._chave)
token_expirado = fixture_token(app._chave, claims_lia,
                              {"iat": int(time.time()) - 600, "exp": int(time.time()) - 300})
esperar_status("token expirado", 401, lambda: app.ler_relatorio(token_expirado, "rel-101"))


In [ ]:
# Testes de limites entre etapas, sempre com relógio controlado.
for _ in range(LIMITE_TENTATIVAS):
    esperar_status("senha errada de Rui", 401, lambda: app.iniciar_login("rui", "errada"))
esperar_status("senha correta durante bloqueio", 429, lambda: app.iniciar_login("rui", SENHA_FICTICIA))
relogio_app.avancar(BLOQUEIO_SEGUNDOS)
desafio_rui = app.iniciar_login("rui", SENHA_FICTICIA)
otp_errado_rui = codigo_invalido(autenticadores_ficticios["rui"], relogio_app())
for _ in range(LIMITE_TENTATIVAS):
    esperar_status("MFA errado de Rui", 401, lambda: app.concluir_mfa(desafio_rui, otp_errado_rui))
novo_desafio_rui = app.iniciar_login("rui", SENHA_FICTICIA)
esperar_status("novo desafio não zera limite MFA", 429,
               lambda: app.concluir_mfa(novo_desafio_rui, autenticadores_ficticios["rui"].at(relogio_app())))
esperar_status("desafio substituído invalidado", 401,
               lambda: app.concluir_mfa(desafio_rui, autenticadores_ficticios["rui"].at(relogio_app())))
relogio_app.avancar(BLOQUEIO_SEGUNDOS)
novo_desafio_rui = app.iniciar_login("rui", SENHA_FICTICIA)
token_rui = app.concluir_mfa(novo_desafio_rui, autenticadores_ficticios["rui"].at(relogio_app()))
verificar("recuperação após pausa MFA", app.ler_relatorio(token_rui, "rel-202")["dono"] == "rui")
esperar_status("propriedade também vale no sentido inverso", 404, lambda: app.ler_relatorio(token_rui, "rel-101"))

desafio_expiravel = app.iniciar_login("bia", SENHA_FICTICIA)
relogio_app.avancar(121)
esperar_status("desafio MFA expirado", 401,
               lambda: app.concluir_mfa(desafio_expiravel, autenticadores_ficticios["bia"].at(relogio_app())))

tokens_equipe = {}
for usuario in ("bia", "eva"):
    desafio = app.iniciar_login(usuario, SENHA_FICTICIA)
    tokens_equipe[usuario] = app.concluir_mfa(desafio, autenticadores_ficticios[usuario].at(relogio_app()))
verificar("analista lê relatório de aluno", app.ler_relatorio(tokens_equipe["bia"], "rel-101")["dono"] == "lia")
esperar_status("analista não administra", 403, lambda: app.listar_usuarios(tokens_equipe["bia"]))
verificar("admin lista contas sem segredos", app.listar_usuarios(tokens_equipe["eva"]) == ["bia", "eva", "lia", "rui"])
app._usuarios["bia"]["role"] = "aluno"  # Simula rebaixamento administrativo confiável.
esperar_status("token com papel antigo invalidado", 401, lambda: app.ler_relatorio(tokens_equipe["bia"], "rel-101"))


In [ ]:
campos_log = {"instante", "etapa", "resultado", "usuario", "recurso"}
verificar("logs só contêm campos previstos", all(set(e) == campos_log for e in app.eventos))
log_serializado = json.dumps(app.eventos, ensure_ascii=False)
segredos_a_nao_registrar = [SENHA_FICTICIA, token_lia, token_rui, desafio_lia,
                           app._chave.hex(), *[g.secret for g in autenticadores_ficticios.values()]]
verificar("senha, tokens, desafios e segredos ausentes dos logs",
          all(segredo not in log_serializado for segredo in segredos_a_nao_registrar))
verificar("eventos cobrem autenticação e autorização",
          {"cadastro", "senha", "mfa", "token", "autorizacao"} <= {e["etapa"] for e in app.eventos})
verificar("recusas de autorização ficam registradas",
          any(e["etapa"] == "autorizacao" and e["resultado"] == "recusado_404" for e in app.eventos))
print("Resumo de eventos, sem credenciais:")
for (etapa, resultado), quantidade in sorted(Counter((e["etapa"], e["resultado"]) for e in app.eventos).items()):
    print(f"  {etapa:12} {resultado:24} {quantidade}")


### 🔎 ANÁLISE
O login só emite JWT depois de verificar o segundo fator. A assinatura protege as claims contra alteração; a política de acesso continua necessária. Um código MFA correto pode ser recusado por replay, desafio expirado ou limite de tentativas. 429 representa limitação de tentativas, distinta de 401/403/404.

**Limites deliberados:** armazenamento volátil, ausência de concorrência e persistência, sem servidor HTTP/TLS, sem fluxo completo de ativação/recuperação de MFA, sem reset de senha, sem cookies, sem revogação individual de JWT, sem rotação de chave e sem proteção de segredos em cofre. Limitação por conta é apenas uma simulação, não uma defesa completa contra automação. Os relógios de TOTP são simulados; o UTC de JWT é real. Reiniciar o kernel apaga contas, desafios e chave. Os logs são inspecionáveis e alteráveis localmente.

### 🎮 DESAFIO
Em uma cópia do notebook, acrescente uma operação `editar_relatorio`: apenas o proprietário ou `admin` podem editar; `analista` continua apenas lendo. Escreva uma matriz de acesso e ao menos seis testes: proprietário, outro aluno, analista, admin, token inválido e recurso inexistente. Inclua os eventos sem registrar conteúdo do relatório.

**Critérios de conclusão:** todos os testes existentes continuam passando; nenhuma função confia em `role`/`dono` enviado pelo cliente; recusas esperadas têm ✅; permissões indevidas levantam exceção. Não transforme a função vulnerável do Bloco 4 em dependência do sistema.

### 💡 DICA
Reutilize `autenticar` e faça a decisão de edição **antes** de mudar o dicionário. Uma autorização de leitura não implica autorização de escrita. Para testar uma nova sessão de uma conta já usada, avance o relógio simulado antes de gerar o TOTP seguinte.

### 🛡️ BLUE TEAM
- Que alertas você criaria para falhas de MFA seguidas de acesso a muitos objetos?
- Como garantir integridade, acesso restrito e prazo de retenção dos logs?
- Qual controle falta para impedir que duas requisições paralelas consumam o mesmo TOTP?
- Que evidências distinguem autenticação bem-sucedida de acesso legítimo?

## Checklist final de aprendizagem

- [ ] Diferencio AuthN e AuthZ e explico 401, 403, 404 e 429.
- [ ] Explico salt individual, custo de PBKDF2 e comparação segura.
- [ ] Distingo proteção contra tentativas offline e online.
- [ ] Reconheço os fatores e as limitações de TOTP contra phishing.
- [ ] Explico tolerância temporal, expiração e prevenção de replay.
- [ ] Distingo sessão stateful, JWT stateless e um modelo híbrido.
- [ ] Valido assinatura, algoritmo permitido, claims, issuer, audience e expiração.
- [ ] Sei que payload legível não significa token confiável ou criptografado.
- [ ] Aplico RBAC, scope e propriedade do recurso com negação por padrão.
- [ ] Escrevo testes de sucesso e recusa que detectam falhas reais.
- [ ] Registro eventos úteis sem guardar credenciais.
- [ ] Identifico o que falta antes de qualquer uso em produção.

## Referências técnicas

Consultadas em 06/10/2026. Os exemplos e desafios deste notebook são próprios; as fontes abaixo fundamentam decisões técnicas.

- [OWASP — Password Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html): escolha de KDF, salt e custo de PBKDF2.
- [OWASP — Authentication Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Authentication_Cheat_Sheet.html): políticas, mensagens e proteção do login.
- [PyOTP — documentação](https://pyauth.github.io/pyotp/): API TOTP, janela e recomendação de rejeitar reutilização.
- [PyJWT — documentação de uso](https://pyjwt.readthedocs.io/en/stable/usage.html): validação de claims e assinatura.
- [PyJWT — referência da API](https://pyjwt.readthedocs.io/en/stable/api.html): allowlist de algoritmos e claims obrigatórias.


In [ ]:
# Se uma verificação falhar, a execução para antes deste resumo.
print(f"✅ AuthLab {VERSAO_MATERIAL}: {len(TESTES)} testes passaram nesta execução.")
print("Fluxo completo: cadastro → senha → MFA → JWT → RBAC/propriedade → eventos.")
print("Reinicie o kernel e execute tudo para repetir a validação com estado novo.")
